# Train, Validation, and Test Split Analysis

1. Examine both splitting methods
2. Check whether information leaks between sets
3. Verify that the sets contain similar types of data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from preprocess import OUT, subsample

df = pd.read_csv(OUT / "manifest.csv")
df = subsample(df, stride=3) # again using stride=3 to reduce the number of samples for faster processing

df.shape

## Split Information

The following columns identify each frame's video, document, document type, background, and assigned dataset splits.

In [ ]:
split_columns = [
    "video_id",
    "model_name",
    "modeltype_name",
    "bg_name",
    "split_video",
    "split_doc"
]

df[split_columns].head()

## Sample Counts

The number of samples in the training, validation, and test sets is compared for both splitting methods. This is to check whether each method creates reasonably sized training, validation, and test sets

In [ ]:
split_order = ["train", "val", "test"]

sample_counts = pd.DataFrame({
    "Video Split": df["split_video"].value_counts(),
    "Document Split": df["split_doc"].value_counts()
}).reindex(split_order)

sample_counts

## Independent Group Counts

The video-based split separates complete video clips, while the document-based split separates complete document models. Frames are not really independent because frames from the same video are extremely similar. Likewise, the same printed document appears in multiple backgrounds

In [ ]:
group_counts = pd.DataFrame({
    "Unique Videos": df.groupby("split_video")["video_id"].nunique(),
    "Unique Documents": df.groupby("split_doc")["model_name"].nunique()
}).reindex(split_order)

group_counts

## Data Leakage Check

No video should appear in multiple video-based splits, and no document should appear in multiple document-based splits.

In [ ]:
video_train = set(df.loc[df["split_video"] == "train", "video_id"])
video_val = set(df.loc[df["split_video"] == "val", "video_id"])
video_test = set(df.loc[df["split_video"] == "test", "video_id"])

document_train = set(df.loc[df["split_doc"] == "train", "model_name"])
document_val = set(df.loc[df["split_doc"] == "val", "model_name"])
document_test = set(df.loc[df["split_doc"] == "test", "model_name"])

leakage_results = pd.DataFrame({
    "Video Overlap": [
        len(video_train & video_val),
        len(video_train & video_test),
        len(video_val & video_test)
    ],
    "Document Overlap": [
        len(document_train & document_val),
        len(document_train & document_test),
        len(document_val & document_test)
    ]
}, index=["Train–Validation", "Train–Test", "Validation–Test"])

leakage_results

## Split Distributions

The following charts compare the percentages of backgrounds and document types in each train, validation, and test set. Sets can have similar sizes but still contain noticeable different data.

In [ ]:
video_background = pd.crosstab(
    df["split_video"],
    df["bg_name"],
    normalize="index"
).reindex(split_order) * 100

document_background = pd.crosstab(
    df["split_doc"],
    df["bg_name"],
    normalize="index"
).reindex(split_order) * 100

video_type = pd.crosstab(
    df["split_video"],
    df["modeltype_name"],
    normalize="index"
).reindex(split_order) * 100

document_type = pd.crosstab(
    df["split_doc"],
    df["modeltype_name"],
    normalize="index"
).reindex(split_order) * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)

video_background.plot(kind="bar", stacked=True, ax=axes[0, 0])
document_background.plot(kind="bar", stacked=True, ax=axes[0, 1])
video_type.plot(kind="bar", stacked=True, ax=axes[1, 0])
document_type.plot(kind="bar", stacked=True, ax=axes[1, 1])

axes[0, 0].set_title("Backgrounds: Video Split")
axes[0, 1].set_title("Backgrounds: Document Split")
axes[1, 0].set_title("Document Types: Video Split")
axes[1, 1].set_title("Document Types: Document Split")

for ax in axes.flat:
    ax.set_xlabel("")
    ax.set_ylabel("Samples (%)")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=0)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("reports/figures/06_split_analysis.png", dpi=140)
plt.show()

## Main Findings

- Both methods create approximately 60/20/20 splits.

- Video groups do not overlap under the video split.

- Document groups do not overlap under the document split.

- Background and document-type distributions remain similar.

- The document-based split is expected to be more difficult because its validation and test documents were entirely absent from training.